# Silver Layer: Order Tables

**Goal:** take the four order-related Bronze tables (`orders`, `order_items`, `order_payments`, `order_reviews`) and make clean, correctly typed versions in Silver.

Same rules as `02_silver_reference`:
- Each Silver table is built from **one** Bronze table only. No joins here, that is Gold's job.
- Nulls are checked and reported, not silently dropped.
- `_rescued_data` is dropped, `_source_file` and `_ingested_at` carry forward.

**Tables in this notebook:** orders, order_items, order_payments, order_reviews

In [0]:
# ============================================================
# SILVER: order tables
# Reads the four Bronze order tables, applies cleaning/casting,
# validates each one, and writes them as managed Delta tables in Silver.
# ============================================================

# All imports live here, once, so every section below can use them
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, initcap, upper

## Pre-flight inspection (read-only)

Before writing any transformation, I look at all four Bronze tables in one go.<br> This keeps the cluster running for as little time as possible, and every decision after this is based on what is actually in the data. Nothing in this section writes anything.

In [0]:
# INSPECTION 1: Peek at each table, its column types, and its row count

order_tables = ["orders", "order_items", "order_payments", "order_reviews"]

for t in order_tables:
    df = spark.table(f"olist_catalog.bronze.{t}")   # open the Bronze table
    print(f"===== {t} =====")
    df.printSchema()                                 # column names + types (all should be string)
    print(f"Bronze row count: {df.count()}")
    display(df.limit(5))                             # a few real rows

In [0]:
# INSPECTION 2: How many nulls are in each column, per table?
# Some nulls are real gaps in the source, so I just want to know where they are.

for t in order_tables:
    df = spark.table(f"olist_catalog.bronze.{t}")
    print(f"===== {t}: null count per column =====")
    for c in df.columns:
        n = df.filter(col(c).isNull()).count()       # count rows where this column is null
        print(f"  {c:<35} {n}")

In [0]:
# INSPECTION 3: Which values exist in the category-style columns,
# and are the key columns unique?

# orders: what order statuses exist?
display(spark
        .table("olist_catalog.bronze.orders")
        .groupBy("order_status").count()
        .orderBy("count", ascending=False)
        )

# order_payments: what payment types exist?
display(spark
        .table("olist_catalog.bronze.order_payments")
        .groupBy("payment_type").count()
        .orderBy("count", ascending=False)
        )

# order_reviews: what review scores exist?
display(spark
        .table("olist_catalog.bronze.order_reviews")
        .groupBy("review_score").count()
        .orderBy("review_score")
        )

# Is each table's key column unique? (total rows vs distinct values)
orders_b  = spark.table("olist_catalog.bronze.orders")
reviews_b = spark.table("olist_catalog.bronze.order_reviews")

print(f"orders:        rows={orders_b.count()},  distinct order_id={orders_b.select('order_id').distinct().count()}")
print(f"order_reviews: rows={reviews_b.count()}, distinct review_id={reviews_b.select('review_id').distinct().count()}")

### Inspection, round 2: follow-ups

Round 1 found that `review_id` is not unique in `order_reviews` (99,224 rows, 98,410 distinct). Before deciding anything, I need to know *why*. I'm also checking the real key of each table, the exact date format, and whether every number-like column casts cleanly. Still read-only, nothing is written.

In [0]:
# INSPECTION 4: Why does review_id repeat in order_reviews?

from pyspark.sql.functions import col
reviews_b = spark.table("olist_catalog.bronze.order_reviews")

# Business columns only, leaving out the lineage columns (_source_file, _ingested_at, _rescued_data)
review_cols = ["review_id", "order_id", "review_score", "review_comment_title",
               "review_comment_message", "review_creation_date", "review_answer_timestamp"]

print(f"Total rows:                         {reviews_b.count()}")
print(f"Distinct review_id:                 {reviews_b.select('review_id').distinct().count()}")
print(f"Distinct (review_id, order_id):     {reviews_b.select('review_id', 'order_id').distinct().count()}")
print(f"Distinct full rows (business cols): {reviews_b.select(review_cols).distinct().count()}")

# How to read the result:
# - If "distinct (review_id, order_id)" = total rows -> same review_id is reused across DIFFERENT orders
# - If "distinct full rows" < total rows -> there are exact duplicate rows

# Peek at a few of the repeated review_ids
dup_ids = (reviews_b.groupBy("review_id").count()
           .filter(col("count") > 1)
           .limit(3))
display(
    reviews_b.join(dup_ids.select("review_id"), "review_id")   # inspection only, not a Silver join
    .select("review_id", "order_id", "review_score", "review_creation_date")
    .orderBy("review_id")
)

In [0]:
# INSPECTION 5: Composite keys
# order_items and order_payments have several rows per order, so order_id alone can't be unique.
# I'm testing the most likely combinations rather than assuming them.

items_b    = spark.table("olist_catalog.bronze.order_items")
payments_b = spark.table("olist_catalog.bronze.order_payments")

print(f"order_items:    rows={items_b.count()}, "
      f"distinct (order_id, order_item_id)={items_b.select('order_id','order_item_id').distinct().count()}")

print(f"order_payments: rows={payments_b.count()}, "
      f"distinct (order_id, payment_sequential)={payments_b.select('order_id','payment_sequential').distinct().count()}")

# For each table, the two numbers should match if that combination is the key.

In [0]:
# INSPECTION 6: Check date/timestamp format
# I need to know whether any value breaks the pattern before casting.

date_cols = {
    "orders":        ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                      "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items":   ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

# Expected pattern: 2017-10-02 10:56:33
pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"

for t, cols in date_cols.items():
    df = spark.table(f"olist_catalog.bronze.{t}")
    for c in cols:
        bad = df.filter(col(c).isNotNull() & ~col(c).rlike(pattern)).count()   # non-null but wrong format
        print(f"{t}.{c}: values NOT matching the pattern: {bad}")

In [0]:
# INSPECTION 6: Check date/timestamp format
# I need to know whether any value breaks the pattern before casting.

date_cols = {
    "orders":        ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                      "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items":   ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

# Expected pattern: 2017-10-02 10:56:33
pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"

for t, cols in date_cols.items():
    df = spark.table(f"olist_catalog.bronze.{t}")
    for c in cols:
        bad = df.filter(col(c).isNotNull() & ~col(c).rlike(pattern)).count()   # non-null but wrong format
        print(f"{t}.{c}: values NOT matching the pattern: {bad}")

In [0]:
# INSPECTION 7: Will the number-like columns cast cleanly?
# cast() turns a bad value into null without an error, so I count values that are
# non-null as text but null after casting. That count should be 0 for each column.

num_checks = {
    "order_items":    ["order_item_id", "price", "freight_value"],
    "order_payments": ["payment_sequential", "payment_installments", "payment_value"],
    "order_reviews":  ["review_score"],
}

for t, cols in num_checks.items():
    df = spark.table(f"olist_catalog.bronze.{t}")
    for c in cols:
        failed = df.filter(col(c).isNotNull() & col(c).cast("double").isNull()).count()
        print(f"{t}.{c}: values that fail to cast to a number: {failed}")

# Range check on money columns: any negatives or zeros?
for t, c in [("order_items", "price"), ("order_items", "freight_value"), ("order_payments", "payment_value")]:
    df = spark.table(f"olist_catalog.bronze.{t}").withColumn("v", col(c).cast("double"))
    stats = df.agg(F.min("v").alias("min"), F.max("v").alias("max"),
                   F.sum(F.when(col("v") == 0, 1).otherwise(0)).alias("zeros")).collect()[0]
    print(f"{t}.{c}: min={stats['min']}, max={stats['max']}, rows equal to 0={stats['zeros']}")

In [0]:
from pyspark.sql.functions import col, trim, initcap, upper, to_timestamp

_____________________________________________________________________

# Silver Layer: Orders

**Goal:** take the raw `orders` table from Bronze and make a clean, correctly typed version of it in Silver.

Bronze holds the data exactly as it arrived, every column is text. The five date columns are still plain text, so Silver is where they become real timestamps.

Orders table transformation list
1. Look at the data first
2. Rename columns where the table name already makes the prefix redundant
3. Cast the five date columns from text to timestamp
4. Check nulls: ID columns, and confirm the cast did not create any new nulls
5. Validate order_id is unique
6. Drop _rescued_data
7. Write to Silver

### Step 1 for orders: Look at the data first

Inspect the raw Bronze orders table before changing anything. Every column should still be a string, carried over from Bronze.

In [0]:
# STEP 1: Look at the data first
df_orders_bronze = spark.table("olist_catalog.bronze.orders")

# Quick peek at real values
display(df_orders_bronze.limit(5))

# Confirm column names and types (everything should still be string, except _ingested_at)
df_orders_bronze.printSchema()

# Row count now, to compare against after every step
print(f"Bronze row count: {df_orders_bronze.count()}")

### Step 2 for orders: Rename the columns

Dropping the order_ prefix where the table name already makes it obvious, but keeping it on order_id since it's the join key to the other order tables and needs to stay unambiguous. customer_id is already unambiguous, so it stays as it is too.

In [0]:
# STEP 2: Rename the columns
df_renamed = (
    df_orders_bronze
    .withColumnRenamed("order_status", "status")
    .withColumnRenamed("order_purchase_timestamp", "purchase_timestamp")
    .withColumnRenamed("order_approved_at", "approved_at")
    .withColumnRenamed("order_delivered_carrier_date", "delivered_carrier_date")
    .withColumnRenamed("order_delivered_customer_date", "delivered_customer_date")
    .withColumnRenamed("order_estimated_delivery_date", "estimated_delivery_date")
    # order_id and customer_id are left as they are on purpose
)

display(df_renamed.limit(5))

### Step 3 for orders: Cast the date columns to timestamp

These five columns arrived as text. As text they can't be sorted by time or subtracted to get a delivery duration. The inspection showed every non-null value matches one format, `yyyy-MM-dd HH:mm:ss`, with 0 mismatches, so one format string can convert all five.

Missing values stay null.

In [0]:
# STEP 3: Cast the date columns to timestamp
ts_format = "yyyy-MM-dd HH:mm:ss"   # e.g. 2017-10-02 10:56:33, confirmed in the inspection

timestamp_cols = ["purchase_timestamp", "approved_at", "delivered_carrier_date",
                  "delivered_customer_date", "estimated_delivery_date"]

df_cast = df_renamed
for c in timestamp_cols:
    df_cast = df_cast.withColumn(c, to_timestamp(col(c), ts_format))   # text -> timestamp

df_cast.printSchema()
display(df_cast.limit(5))

### Step 4 for orders: Check for nulls

Two checks. First, the ID columns should never be null. Second, `cast()` turns a value it can't read into null without an error, so I compare each date column's null count before and after the cast. If the counts are equal, the cast created no new nulls, and every null is a real gap from the source.

In [0]:
# STEP 4: Check for nulls

# Check 1: IDs should never be null
null_ids = df_cast.filter(col("order_id").isNull() | col("customer_id").isNull()).count()
print(f"Rows with a null order_id or customer_id: {null_ids}")

# Check 2: null counts in each date column, Bronze (text) vs Silver (timestamp)
# Left side is the new column name, right side is the original Bronze name
ts_name_map = {
    "purchase_timestamp":      "order_purchase_timestamp",
    "approved_at":             "order_approved_at",
    "delivered_carrier_date":  "order_delivered_carrier_date",
    "delivered_customer_date": "order_delivered_customer_date",
    "estimated_delivery_date": "order_estimated_delivery_date",
}

print(f"\n{'Column':<28} {'Bronze nulls':>13} {'Silver nulls':>13}")
for new_name, old_name in ts_name_map.items():
    bronze_nulls = df_orders_bronze.filter(col(old_name).isNull()).count()
    silver_nulls = df_cast.filter(col(new_name).isNull()).count()
    print(f"{new_name:<28} {bronze_nulls:>13} {silver_nulls:>13}")

# Each pair of numbers should match. Expected from the inspection: 0, 160, 1783, 2965, 0

In [0]:
# STEP 4 (continued): Do the missing delivery dates line up with order status?
# An order that was canceled or is still being processed can't have a delivery date yet,
# so I expect the nulls to be concentrated in those statuses, not in "delivered".

display(
    df_cast.filter(col("delivered_customer_date").isNull())
    .groupBy("status").count()
    .orderBy("count", ascending=False)
)

### Step 5 for orders: Validate uniqueness

order_id should be unique, one row per order. The inspection already showed 99,441 = 99,441 in Bronze, and I check it again here so Silver is proven on its own.

In [0]:
# STEP 5: Validate uniqueness
total_rows = df_cast.count()
distinct_order_id = df_cast.select("order_id").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct order_id: {distinct_order_id}")
# These two numbers should match exactly. If they don't, order_id has duplicates.

### Step 6 for orders: Drop _rescued_data

Same reasoning as every other table. It only catches malformed rows during the Bronze load, and the inspection showed it is null on all 99,441 rows. It's a Bronze diagnostic, not something Silver needs to carry forward.

In [0]:
# STEP 6: Drop _rescued_data
df_cast = df_cast.drop("_rescued_data")

display(df_cast.limit(5))

### Step 7 for orders: Write to Silver

Last step for this table. _source_file and _ingested_at carry forward unchanged, so I can always see which Bronze load produced a Silver row.

In [0]:
# STEP 7: Write to Silver
df_cast.write.format("delta").mode("overwrite").saveAsTable("olist_catalog.silver.orders")

print("Written: olist_catalog.silver.orders")

_____________________________________________________________________

# Silver Layer: Order Items

**Goal:** take the raw `order_items` table from Bronze and make a clean, correctly typed version of it in Silver.

This table has one row per item in an order. An order with three items has three rows. It also holds the money columns (`price`, `freight_value`), so it needs more care with types than the reference tables.

No rename step here: the column names are already clear, and `order_id`, `product_id` and `seller_id` are join keys that need to stay unambiguous.

Order items table transformation list
1. Look at the data first
2. Cast types: order_item_id to integer, shipping_limit_date to timestamp, price and freight_value to decimal(10,2)
3. Check nulls in every business column (the cast must not create new nulls)
4. Validate the key is unique: one row per (order_id, order_item_id) pair
5. Sanity-check the money columns (ranges, and the rows with zero freight)
6. Drop _rescued_data
7. Write to Silver

### Step 1 for order_items: Look at the data first

Inspect the raw Bronze order_items table before changing anything. Every column should still be a string, carried over from Bronze.

In [0]:
# STEP 1: Look at the data first
df_items_bronze = spark.table("olist_catalog.bronze.order_items")

# Quick peek at real values
display(df_items_bronze.limit(5))

# Confirm column names and types (everything should still be string, except _ingested_at)
df_items_bronze.printSchema()

# Row count now, to compare against after every step
print(f"Bronze row count: {df_items_bronze.count()}")

### Step 2 for order_items: Cast the columns to proper types

- `order_item_id` is a counter (1, 2, 3 ...), so **integer**.
- `shipping_limit_date` is a date and time, so **timestamp**. The inspection showed every value matches `yyyy-MM-dd HH:mm:ss`.
- `price` and `freight_value` are money, so **decimal(10,2)**: up to 10 digits in total, exactly 2 after the decimal point. Unlike `double`, a decimal stores cents exactly, so adding up thousands of prices in Gold won't pick up tiny rounding errors. The largest price is 6,735.00, so it fits easily.

The inspection also showed 0 values failing to cast, so I expect no new nulls.

In [0]:
# STEP 2: Cast the columns to proper types
ts_format = "yyyy-MM-dd HH:mm:ss"   # e.g. 2017-09-19 09:45:35, confirmed in the inspection

df_cast = (
    df_items_bronze
    .withColumn("order_item_id", col("order_item_id").cast("integer"))                  # "1" -> 1
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"), ts_format))
    .withColumn("price", col("price").cast("decimal(10,2)"))                            # "58.90" -> 58.90
    .withColumn("freight_value", col("freight_value").cast("decimal(10,2)"))            # "13.29" -> 13.29
)

df_cast.printSchema()
display(df_cast.limit(5))

### Step 3 for order_items: Check for nulls

`cast()` turns a value it can't read into null without an error. Bronze had 0 nulls in every business column, so every column should still have 0 nulls after the casts. Any number above 0 here means a cast failed.

In [0]:
# STEP 3: Check for nulls in every business column
# (lineage columns and _rescued_data are left out: _rescued_data is dropped in Step 6)

business_cols = ["order_id", "order_item_id", "product_id", "seller_id",
                 "shipping_limit_date", "price", "freight_value"]

for c in business_cols:
    null_count = df_cast.filter(col(c).isNull()).count()
    print(f"Nulls in {c:<22} {null_count}")

# Every line should print 0. Bronze had 0 nulls in all of these.

### Step 4 for order_items: Validate uniqueness

`order_id` alone repeats here, because one order can have several items. The inspection showed the pair (`order_id`, `order_item_id`) is unique. I check it again so Silver is proven on its own.

In [0]:
# STEP 4: Validate uniqueness of the (order_id, order_item_id) pair
total_rows = df_cast.count()
distinct_pairs = df_cast.select("order_id", "order_item_id").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct (order_id, order_item_id) pairs: {distinct_pairs}")
# These two numbers should match exactly (112650). If they don't, the pair is not a safe key.

### Step 5 for order_items: Sanity-check the money columns

Two things I want to see after the decimal cast: the ranges look sensible, and the rows with a freight of 0.00 are still there. The inspection found 383 of them. Free shipping is a real thing, so they are legitimate data and Silver keeps them.

In [0]:
# STEP 5: Sanity-check the money columns

# Smallest and largest value in each money column
df_cast.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.min("freight_value").alias("min_freight"),
    F.max("freight_value").alias("max_freight"),
).show()
# Expected from the inspection: min_price 0.85, max_price 6735.00, min_freight 0.00, max_freight 409.68

# How many rows have zero freight? (free shipping, kept on purpose)
zero_freight = df_cast.filter(col("freight_value") == 0).count()
print(f"Rows with freight_value = 0: {zero_freight}")   # expected: 383

### Step 6 for order_items: Drop _rescued_data

Same reasoning as every other table. It only catches malformed rows during the Bronze load, and the inspection showed it is null on all 112,650 rows. It's a Bronze diagnostic, not something Silver needs to carry forward.

In [0]:
# STEP 6: Drop _rescued_data
df_cast = df_cast.drop("_rescued_data")

display(df_cast.limit(5))

### Step 7 for order_items: Write to Silver

Last step for this table. _source_file and _ingested_at carry forward unchanged.

In [0]:
# STEP 7: Write to Silver
df_cast.write.format("delta").mode("overwrite").saveAsTable("olist_catalog.silver.order_items")

print("Written: olist_catalog.silver.order_items")

_____________________________________________________________________

# Silver Layer: Order Payments

**Goal:** take the raw `order_payments` table from Bronze and make a clean, correctly typed version of it in Silver.

One order can be paid in several parts (for example part credit card, part voucher), so this table has one row per payment part, not one row per order.

No rename step here: after dropping the `payment_` prefix, columns called `type` and `value` would be ambiguous once this table is joined to others in Gold, so the names stay as they are.

Order payments table transformation list
1. Look at the data first
2. Cast types: payment_sequential and payment_installments to integer, payment_value to decimal(10,2)
3. Check nulls in every business column (the cast must not create new nulls)
4. Validate the key is unique: one row per (order_id, payment_sequential) pair
5. Check the payment_type values, and sanity-check the numbers (including rows with a payment value of 0)
6. Drop _rescued_data
7. Write to Silver

### Step 1 for order_payments: Look at the data first

Inspect the raw Bronze order_payments table before changing anything. Every column should still be a string, carried over from Bronze.

In [0]:
# STEP 1: Look at the data first
df_payments_bronze = spark.table("olist_catalog.bronze.order_payments")

# Quick peek at real values
display(df_payments_bronze.limit(5))

# Confirm column names and types (everything should still be string, except _ingested_at)
df_payments_bronze.printSchema()

# Row count now, to compare against after every step
print(f"Bronze row count: {df_payments_bronze.count()}")

### Step 2 for order_payments: Cast the columns to proper types

- `payment_sequential` is a counter (1st payment part, 2nd payment part ...), so **integer**.
- `payment_installments` is a count of instalments, so **integer**.
- `payment_value` is money, so **decimal(10,2)**, for the same reason as `price` in order_items: it stores cents exactly, so sums don't drift. The largest value is 13,664.08, so it fits easily.
- `payment_type` is a category label and stays as **text**.

The inspection showed 0 values failing to cast, so I expect no new nulls.

In [0]:
# STEP 2: Cast the columns to proper types
df_cast = (
    df_payments_bronze
    .withColumn("payment_sequential", col("payment_sequential").cast("integer"))        # "1" -> 1
    .withColumn("payment_installments", col("payment_installments").cast("integer"))    # "8" -> 8
    .withColumn("payment_value", col("payment_value").cast("decimal(10,2)"))            # "99.33" -> 99.33
    # payment_type stays as text on purpose
)

df_cast.printSchema()
display(df_cast.limit(5))

### Step 3 for order_payments: Check for nulls

`cast()` turns a value it can't read into null without an error. Bronze had 0 nulls in every business column, so every column should still have 0 nulls after the casts. Any number above 0 here means a cast failed.

In [0]:
# STEP 3: Check for nulls in every business column

business_cols = ["order_id", "payment_sequential", "payment_type",
                 "payment_installments", "payment_value"]

for c in business_cols:
    null_count = df_cast.filter(col(c).isNull()).count()
    print(f"Nulls in {c:<22} {null_count}")

# Every line should print 0. Bronze had 0 nulls in all of these.

### Step 4 for order_payments: Validate uniqueness

`order_id` alone repeats here, because one order can have several payment parts. The inspection showed the pair (`order_id`, `payment_sequential`) is unique. I check it again so Silver is proven on its own.

In [0]:
# STEP 4: Validate uniqueness of the (order_id, payment_sequential) pair
total_rows = df_cast.count()
distinct_pairs = df_cast.select("order_id", "payment_sequential").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct (order_id, payment_sequential) pairs: {distinct_pairs}")
# These two numbers should match exactly (103886). If they don't, the pair is not a safe key.

### Step 5 for order_payments: Check payment types and sanity-check the numbers

Three things to look at, none of which change the data:
- Which `payment_type` values exist. The inspection found 5, including 3 rows labelled `not_defined`.
- The smallest and largest value in each numeric column, to catch anything odd such as a negative number.
- The rows with a `payment_value` of 0. The inspection found 9.

These are real values from the source, so Silver keeps all of them. What to do with them is a decision for Gold or the analyst.

In [0]:
# STEP 5: Check payment types and sanity-check the numbers

# Which payment types exist, and how many rows each?
display(df_cast.groupBy("payment_type").count().orderBy("count", ascending=False))
# Expected from the inspection: credit_card, boleto, voucher, debit_card, not_defined (3 rows)

# Smallest and largest value in each numeric column
df_cast.agg(
    F.min("payment_sequential").alias("min_seq"),
    F.max("payment_sequential").alias("max_seq"),
    F.min("payment_installments").alias("min_installments"),
    F.max("payment_installments").alias("max_installments"),
    F.min("payment_value").alias("min_value"),
    F.max("payment_value").alias("max_value"),
).show()
# Expected from the inspection: min_value 0.00, max_value 13664.08. Nothing should be negative.

# The rows with a payment value of 0, with their payment type
zero_value_rows = df_cast.filter(col("payment_value") == 0)
print(f"Rows with payment_value = 0: {zero_value_rows.count()}")   # expected: 9
display(zero_value_rows.select("order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"))

### Step 6 for order_payments: Drop _rescued_data

Same reasoning as every other table. It only catches malformed rows during the Bronze load, and the inspection showed it is null on all 103,886 rows. It's a Bronze diagnostic, not something Silver needs to carry forward.

In [0]:
# STEP 6: Drop _rescued_data
df_cast = df_cast.drop("_rescued_data")

display(df_cast.limit(5))

### Step 7 for order_payments: Write to Silver

Last step for this table. _source_file and _ingested_at carry forward unchanged.

In [0]:
# STEP 7: Write to Silver
df_cast.write.format("delta").mode("overwrite").saveAsTable("olist_catalog.silver.order_payments")

print("Written: olist_catalog.silver.order_payments")

_____________________________________________________________________

# Silver Layer: Order Reviews

**Goal:** take the raw `order_reviews` table from Bronze and make a clean, correctly typed version of it in Silver.

This is the only table with free text written by customers (`review_comment_title`, `review_comment_message`). Back in Bronze the file needed special parsing (`multiLine`) because review text can contain line breaks and commas, and all 99,224 rows loaded with 0 rescued.

Two decisions for this table:
- `review_id` is **not** unique on its own. The inspection showed 99,224 rows but 98,410 distinct `review_id`, and that the pair (`review_id`, `order_id`) is unique, with no exact duplicate rows. So nothing is dropped, and the pair is what gets validated.
- The review text is left exactly as it arrived. It's customer content, not something Silver should reformat.

No rename step: names like `score` would be ambiguous once this table is joined to others in Gold.

Order reviews table transformation list
1. Look at the data first
2. Cast types: review_score to integer, the two date columns to timestamp
3. Check nulls (the cast must not create new nulls; the two comment columns are expected to have many)
4. Validate the key is unique: one row per (review_id, order_id) pair
5. Sanity-check the values (score range, empty comment text, answer date before creation date)
6. Drop _rescued_data
7. Write to Silver

### Step 1 for order_reviews: Look at the data first

Inspect the raw Bronze order_reviews table before changing anything. Every column should still be a string, carried over from Bronze.

In [0]:
# STEP 1: Look at the data first
df_reviews_bronze = spark.table("olist_catalog.bronze.order_reviews")

# Quick peek at real values
display(df_reviews_bronze.limit(5))

# Confirm column names and types (everything should still be string, except _ingested_at)
df_reviews_bronze.printSchema()

# Row count now, to compare against after every step
print(f"Bronze row count: {df_reviews_bronze.count()}")

### Step 2 for order_reviews: Cast the columns to proper types

- `review_score` is a whole number from 1 to 5, so **integer**. The inspection showed only the values 1 to 5, and 0 values failing to cast.
- `review_creation_date` and `review_answer_timestamp` are dates and times, so **timestamp**. Every value matches `yyyy-MM-dd HH:mm:ss`, with 0 mismatches.
- `review_id`, `order_id` and the two comment columns stay as **text**.

In [0]:
# STEP 2: Cast the columns to proper types
ts_format = "yyyy-MM-dd HH:mm:ss"   # e.g. 2018-01-18 21:46:59, confirmed in the inspection

df_cast = (
    df_reviews_bronze
    .withColumn("review_score", col("review_score").cast("integer"))                              # "4" -> 4
    .withColumn("review_creation_date", to_timestamp(col("review_creation_date"), ts_format))
    .withColumn("review_answer_timestamp", to_timestamp(col("review_answer_timestamp"), ts_format))
    # review_comment_title and review_comment_message stay exactly as they arrived
)

df_cast.printSchema()
display(df_cast.limit(5))

### Step 3 for order_reviews: Check for nulls

Most customers leave a score without writing anything, so the two comment columns are expected to have many nulls. Those are genuine gaps and stay null. The other five columns had 0 nulls in Bronze, so they should still have 0 after the casts. Any number above 0 there means a cast failed.

In [0]:
# STEP 3: Check for nulls in every business column

business_cols = ["review_id", "order_id", "review_score",
                 "review_comment_title", "review_comment_message",
                 "review_creation_date", "review_answer_timestamp"]

for c in business_cols:
    null_count = df_cast.filter(col(c).isNull()).count()
    print(f"Nulls in {c:<26} {null_count}")

# Expected from the inspection:
# 0 for review_id, order_id, review_score and both date columns
# 87656 for review_comment_title, 58247 for review_comment_message (real gaps, kept as null)

### Step 4 for order_reviews: Validate uniqueness

`review_id` repeats across different orders, so on its own it is not a key. The inspection showed the pair (`review_id`, `order_id`) is unique. I check the pair again so Silver is proven on its own, and I print the `review_id` count as well, so the difference stays visible.

In [0]:
# STEP 4: Validate uniqueness of the (review_id, order_id) pair
total_rows = df_cast.count()
distinct_pairs = df_cast.select("review_id", "order_id").distinct().count()
distinct_review_id = df_cast.select("review_id").distinct().count()
distinct_order_id = df_cast.select("order_id").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct (review_id, order_id) pairs: {distinct_pairs}")   # should match total rows (99224)
print(f"Distinct review_id alone: {distinct_review_id}")           # expected 98410, NOT a key
print(f"Distinct order_id (for information): {distinct_order_id}")
# Distinct order_id tells me how many orders have a review.
# If it is lower than the row count, some orders have more than one review, worth remembering in Gold.

### Step 5 for order_reviews: Sanity-check the values

Three checks, none of which change the data:
- The review scores should be only 1 to 5, with the same counts as the inspection.
- Comment text can be `null` (no comment) or an empty string (a comment of nothing). I count the second kind, to know whether they exist.
- A review should not be answered before it was created. I count any row where that happens.

Anything found here is reported, not dropped. Silver doesn't know what Gold needs yet.

In [0]:
# STEP 5: Sanity-check the values

# Review score counts (inspection: 1=11424, 2=3151, 3=8179, 4=19142, 5=57328)
display(df_cast.groupBy("review_score").count().orderBy("review_score"))

# Empty-string comments (not null, but nothing in them after trimming spaces)
empty_title   = df_cast.filter(col("review_comment_title").isNotNull()   & (trim(col("review_comment_title")) == "")).count()
empty_message = df_cast.filter(col("review_comment_message").isNotNull() & (trim(col("review_comment_message")) == "")).count()
print(f"Empty-string titles: {empty_title}")
print(f"Empty-string messages: {empty_message}")

# Answered before created?
answered_before_created = df_cast.filter(col("review_answer_timestamp") < col("review_creation_date")).count()
print(f"Rows where the answer timestamp is before the creation date: {answered_before_created}")

### Step 6 for order_reviews: Drop _rescued_data

Same reasoning as every other table. It only catches malformed rows during the Bronze load. For this table it matters most of all: it was null on all 99,224 rows, which confirms the `multiLine` parsing handled the line breaks and commas inside review text. It's a Bronze diagnostic, not something Silver needs to carry forward.

In [0]:
# STEP 6: Drop _rescued_data
df_cast = df_cast.drop("_rescued_data")

display(df_cast.limit(5))

### Step 7 for order_reviews: Write to Silver

Last step for this table, and the last table in silver-order-tables. _source_file and _ingested_at carry forward unchanged.

In [0]:
# STEP 7: Write to Silver
df_cast.write.format("delta").mode("overwrite").saveAsTable("olist_catalog.silver.order_reviews")

print("Written: olist_catalog.silver.order_reviews")

_____________________________________________________________________

### Final validation pass: silver-order-tables

One combined check across all four tables: row counts against Bronze. No table should have lost or gained a row, because Silver here only renames, types and validates. It does not filter. A second check confirms each table exists in olist_catalog.silver.

In [0]:
# FINAL VALIDATION: silver-order-tables branch

tables = ["orders", "order_items", "order_payments", "order_reviews"]

print(f"{'Table':<20} {'Bronze rows':>12} {'Silver rows':>12} {'Match':>8}")
print("-" * 56)

for t in tables:
    bronze_count = spark.table(f"olist_catalog.bronze.{t}").count()
    silver_count = spark.table(f"olist_catalog.silver.{t}").count()
    match = "✅" if bronze_count == silver_count else "❌"
    print(f"{t:<20} {bronze_count:>12} {silver_count:>12} {match:>8}")

# Spot check: all four tables should now be listed in the silver schema
display(spark.sql("SHOW TABLES IN olist_catalog.silver"))